# 지식 그래프 실습

**Knowledge Graph · RDF · 트리플**

대상과 대상 사이의 관계를 노드와 간선으로 저장한 그래프 형태의 지식 표현. 흩어진 데이터를 연결해 질의할 수 있게 한다.

소재 분야에서 이해하기: 조성–공정–물성–문헌을 연결해 특정 물성을 낸 공정 경로를 추적한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [W3C RDF 데이터 모델](https://www.w3.org/RDF/)

## 1. 트리플로 저장하기

지식 그래프의 최소 단위는 (주어, 관계, 목적어) 세 쌍입니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

# 개념 확인용 작은 지식 그래프. 실제 데이터베이스가 아닙니다.
TRIPLES = [
    # 시료 - 조성
    ('sample:A1', 'hasElement', 'element:Fe'), ('sample:A1', 'hasElement', 'element:Cr'),
    ('sample:A2', 'hasElement', 'element:Fe'), ('sample:A2', 'hasElement', 'element:Ni'),
    ('sample:B1', 'hasElement', 'element:Ti'), ('sample:B1', 'hasElement', 'element:Al'),
    # 시료 - 공정
    ('sample:A1', 'madeBy', 'process:P780'), ('sample:A2', 'madeBy', 'process:P860'),
    ('sample:B1', 'madeBy', 'process:P780'),
    ('process:P780', 'temperatureC', '780'), ('process:P860', 'temperatureC', '860'),
    ('process:P780', 'processType', 'type:Sintering'), ('process:P860', 'processType', 'type:Sintering'),
    # 시료 - 물성
    ('sample:A1', 'hardnessHV', '431'), ('sample:A2', 'hardnessHV', '455'),
    ('sample:B1', 'hardnessHV', '388'),
    # 시료 - 문헌
    ('sample:A1', 'reportedIn', 'paper:10.1000/aaa'), ('sample:A2', 'reportedIn', 'paper:10.1000/aaa'),
    ('sample:B1', 'reportedIn', 'paper:10.1000/bbb'),
    ('paper:10.1000/aaa', 'year', '2026'), ('paper:10.1000/bbb', 'year', '2025'),
]
print('트리플', len(TRIPLES), '개')

In [ ]:
from collections import defaultdict

out_edges = defaultdict(list)
in_edges = defaultdict(list)
for subject, predicate, obj in TRIPLES:
    out_edges[subject].append((predicate, obj))
    in_edges[obj].append((predicate, subject))

print('sample:A1 에서 나가는 관계:')
for predicate, obj in out_edges['sample:A1']:
    print('   %-12s -> %s' % (predicate, obj))
print('\nprocess:P780 을 가리키는 것들:', [s for _, s in in_edges['process:P780']])

## 2. 관계 경로로 묻기

표라면 여러 번 조인해야 하는 질문을 경로 한 줄로 물을 수 있습니다.

In [ ]:
def follow(start, path):
    """관계 이름을 순서대로 따라가며 도달하는 노드를 모읍니다."""
    current = {start}
    for predicate in path:
        nxt = set()
        for node in current:
            nxt |= {obj for name, obj in out_edges[node] if name == predicate}
        current = nxt
    return current

print('A1 의 소성 온도:', follow('sample:A1', ['madeBy', 'temperatureC']))
print('A1 이 보고된 논문의 연도:', follow('sample:A1', ['reportedIn', 'year']))

# Fe 를 포함하고 780 C 로 소성한 시료
def samples_with(element, temperature):
    with_element = {s for _, s in in_edges['element:' + element]}
    at_temp = set()
    for node, edges in out_edges.items():
        if node.startswith('sample:') and follow(node, ['madeBy', 'temperatureC']) == {temperature}:
            at_temp.add(node)
    return sorted(with_element & at_temp)

print('\nFe 포함 + 780 C 소성:', samples_with('Fe', '780'))

## 3. 그래프 구조 보기

어떤 노드가 여러 사실을 잇는 중심인지 연결 수로 확인합니다.

In [ ]:
nodes = sorted(set([s for s, _, _ in TRIPLES]) | set([o for _, _, o in TRIPLES]))
index = {node: position for position, node in enumerate(nodes)}
adjacency = np.zeros((len(nodes), len(nodes)))
for subject, _, obj in TRIPLES:
    adjacency[index[subject], index[obj]] = 1
    adjacency[index[obj], index[subject]] = 1

degree = adjacency.sum(1)
order = np.argsort(-degree)[:6]
for position in order:
    print('%-22s 연결 %d' % (nodes[position], int(degree[position])))

plt.imshow(adjacency, cmap='Greys')
plt.xticks([]); plt.yticks([]); plt.title('adjacency of the small graph'); plt.show()

## 4. 데이터를 합치고 빈 연결을 추정하기

In [ ]:
# 같은 식별자를 쓰면 다른 출처의 사실이 자동으로 이어집니다.
EXTRA = [('sample:A2', 'hasElement', 'element:Cr'), ('sample:B2', 'hasElement', 'element:Ti'),
         ('sample:B2', 'madeBy', 'process:P860'), ('sample:B2', 'hardnessHV', '402')]
for triple in EXTRA:
    subject, predicate, obj = triple
    out_edges[subject].append((predicate, obj)); in_edges[obj].append((predicate, subject))
print('B2 가 합쳐진 뒤 P860 을 쓰는 시료:', sorted(s for _, s in in_edges['process:P860']))

# 공통 이웃이 많은 쌍은 아직 기록되지 않은 관계의 후보가 됩니다.
def neighbours(node):
    return {obj for _, obj in out_edges[node]} | {s for _, s in in_edges[node]}

samples = sorted(n for n in set(list(out_edges) + list(in_edges)) if n.startswith('sample:'))
print('\n공통 이웃 수로 본 시료 유사도:')
for i, left in enumerate(samples):
    for right in samples[i + 1:]:
        shared = neighbours(left) & neighbours(right)
        if shared:
            print('   %-11s %-11s 공통 %d개 %s' % (left, right, len(shared), sorted(shared)))
print('\n이것이 지식 그래프 기반 추천·링크 예측의 가장 단순한 형태입니다.')

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#knowledge-graph)을 여세요.